# Preprocessing pipeline

Runs the full pipeline on a BCI2000 recording:
1. Load with `load_bci2k`
2. Bandpass filter [1–55 Hz], 4th-order Butterworth
3. Re-reference to average
4. ICA (extended Infomax, 24 components)
5. ICLabel classification
6. Plot first 2 components with ICLabel annotations

In [ ]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

PROJECT_ROOT = Path("../..").resolve()
SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from eeg.io import load_bci2k
from eeg.inspect import print_summary
from eeg.preprocessing import bandpass_filter, rereference_average, run_ica, label_components
from eeg.viz import plot_ica_components, plot_stim_channel, plot_channels, plot_channels_psd

DATA_FILE = PROJECT_ROOT / "data/eeg/raw/MET000bGridFixedS001R02.dat"

## 1. Load

In [ ]:
raw = load_bci2k(DATA_FILE)
print_summary(raw)

In [ ]:
plot_stim_channel(raw, 600)

## 2. Bandpass filter

In [ ]:
L_FREQ = 1.0
H_FREQ = 55.0
FILTER_ORDER = 4

bandpass_filter(raw, l_freq=L_FREQ, h_freq=H_FREQ, order=FILTER_ORDER)

## 3. Re-reference to average

In [ ]:
rereference_average(raw)

In [ ]:
CHANNELS = ["Oz", "O1", "Pz"]  # change to any channel names
DURATION = 100.0  # seconds to show

plot_channels(raw, CHANNELS, duration=DURATION)

In [ ]:
FMAX = 55.0  # Hz

plot_channels_psd(raw, CHANNELS, fmax=FMAX)

## 4. ICA

Takes ~60–90 seconds.

In [ ]:
N_COMPONENTS = 24

ica = run_ica(raw, n_components=N_COMPONENTS)
print(ica)

## 5. ICLabel

In [ ]:
labels = label_components(raw, ica)

for i, (label, proba) in enumerate(zip(labels["labels"], labels["y_pred_proba"])):
    print(f"IC {i:02d}: {label} ({proba.max():.0%})")

## 6. Plot first 2 components

Change `COMPONENT_INDICES` to inspect different components.

In [ ]:
COMPONENT_INDICES = [0, 1, 2, 3, 4]

figs = plot_ica_components(raw, ica, COMPONENT_INDICES, labels=labels)
for fig in figs:
    fig.show()